In [2]:
!pip install Bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.3/321.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Bio.Seq import Seq
from Bio import SeqIO
from Bio import Align
from Bio import AlignIO
from Bio.Align import substitution_matrices
from Bio.Data import IUPACData
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.SeqRecord import SeqRecord
from Bio import pairwise2
from Bio.pairwise2 import format_alignment
from pathlib import Path
import os
import h5py
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model
from sklearn.metrics import precision_recall_fscore_support, f1_score

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [3]:
# Mount your personal Google drive
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
test_data_path = Path('drive/MyDrive/BD/biological_data_pfp/test')

baseline_data_path = Path('.drive/MyDrive/BD/biological_data_pfp/baseline')

# Test set preparation

In [5]:
# Extracting test_ids.txt
with open(test_data_path / 'test_ids.txt', 'r') as file:
    test_ids = file.read().splitlines()

In [6]:
data_list = []

with h5py.File(test_data_path / "test_embeddings.h5", "r") as f:
    for dataset_name in f.keys():
        dataset = f[dataset_name][:]
        data_list.append([dataset_name, dataset])

test_embeddings = pd.DataFrame(data_list, columns=["ID", "embeddings"])

test_embeddings.head()

,ID,embeddings
0,A0A0B4JCV4,"[0.00979, -0.03973, 0.03653, -0.006447, -0.040..."
1,A0A0B4KHT0,"[0.02786, -0.01154, 0.008865, -0.01765, 0.0073..."
2,A0A0B4P506,"[0.01643, 0.01802, 0.03702, -0.0591, 0.0356, 0..."
3,A0A0G2K1A2,"[0.00882, 0.0835, -0.001374, -0.0003645, -0.06..."
4,A0A0G2K1V4,"[0.0659, 0.0929, -0.001803, 0.0226, 0.0383, 0...."


In [7]:
test_protein2ipr = pd.read_csv(test_data_path / 'test_protein2ipr.dat', sep='\t')

# Rename Protein_ID and aspect columns
test_protein2ipr.columns = ['ID', 'ipr', 'domain', 'familyID', 'start', 'end']

# Remove 'domain' that is useless and heavy
test_protein2ipr.drop('domain', axis=1)

test_protein2ipr.head()

,ID,ipr,domain,familyID,start,end
0,A0A0B4JCV4,IPR039915,TACC family,PTHR13924,38,1206
1,A0A0B4KHT0,IPR000315,B-box-type zinc finger,PF00643,177,219
2,A0A0B4KHT0,IPR000315,B-box-type zinc finger,PF00643,236,274
3,A0A0B4KHT0,IPR000315,B-box-type zinc finger,PS50119,173,220
4,A0A0B4KHT0,IPR000315,B-box-type zinc finger,PS50119,235,282


In [8]:
# Group by 'ID' and aggregate other columns into lists
test_protein2ipr_grouped = test_protein2ipr.groupby('ID').agg(lambda x: tuple(x)).reset_index()

print(f"Test protein2ipr ({test_protein2ipr.shape}):")

Test protein2ipr ((11263, 6)):


In [9]:
combined_test = pd.merge(test_embeddings, test_protein2ipr_grouped, on='ID', how='left')

missing_rows = combined_test[combined_test['ipr'].isna()].shape[0]
print(f"Number of rows missing from train_protein2ipr_grouped: {missing_rows}")

print(f"Combined DataFrame shape: {combined_test.shape}")

Number of rows missing from train_protein2ipr_grouped: 19
Combined DataFrame shape: (1000, 7)


In [17]:
test_protein_ids = combined_test['ID'].tolist()
X_test = np.stack(combined_test['embeddings'].values).astype('float32')

In [25]:
def predict_raw_candidates(model, X_data, mlb_classes, min_threshold=0.01):
    """
    Returns ALL candidates above the minimum threshold, without sorting or truncating yet.
    """
    print(f"Calculating raw probabilities (Threshold > {min_threshold})...")
    # Large batch size for speed
    probs = model.predict(X_data, batch_size=512, verbose=1)

    all_candidates = []

    for i, row_probs in enumerate(probs):
        # 1. Fast filtering with NumPy
        high_idx = np.where(row_probs > min_threshold)[0]

        # 2. If empty, take at least the best one (for safety)
        if len(high_idx) == 0:
            high_idx = np.array([np.argmax(row_probs)])

        selected_probs = row_probs[high_idx]

        # 3. Build the list of tuples (Term, Score)
        protein_cands = []
        for idx, score in zip(high_idx, selected_probs):
            term = mlb_classes[idx]
            protein_cands.append((term, float(score)))

        all_candidates.append(protein_cands)

    return all_candidates

In [12]:
best_nn2_model_CC = load_model('best_nn2_model_CC.keras')
best_nn2_model_BP = load_model('best_nn2_model_BP.keras')
best_nn3_model_MF = load_model('best_nn3_model_MF.keras')

In [14]:
import numpy as np
import pickle

# 1. Loading MultiLabelBinarizers (Ensure files exist)
with open("mlb_cc.pkl", "rb") as f:
    mlb_cc = pickle.load(f)
with open("mlb_bp.pkl", "rb") as f:
    mlb_bp = pickle.load(f)
with open("mlb_mf.pkl", "rb") as f:
    mlb_mf = pickle.load(f)

In [26]:
# Used to save memory by filtering out absolute zeros.
# 0.01 is very conservative (keeps almost anything with minimal plausibility).
#usefull because of cafa eval multiple thresholds evaluation
MIN_CONFIDENCE = 0.01

raw_cc = predict_raw_candidates(best_nn2_model_CC, X_test, mlb_cc.classes_, min_threshold=MIN_CONFIDENCE)

raw_bp = predict_raw_candidates(best_nn2_model_BP, X_test, mlb_bp.classes_, min_threshold=MIN_CONFIDENCE)

raw_mf = predict_raw_candidates(best_nn3_model_MF, X_test, mlb_mf.classes_, min_threshold=MIN_CONFIDENCE)

output_file = "submission.tsv"
MAX_TERMS_TOTAL = 1500

with open(output_file, "w") as f:
    # Iterate over each protein
    for i, pid in enumerate(test_protein_ids):

        # Merge everything into a single pool
        combined_preds = raw_cc[i] + raw_bp[i] + raw_mf[i]

        # The highest score wins, regardless of whether it's BP, MF, or CC
        combined_preds.sort(key=lambda x: x[1], reverse=True)

        # Truncate to the absolute Top-1500
        if len(combined_preds) > MAX_TERMS_TOTAL:
            final_preds = combined_preds[:MAX_TERMS_TOTAL]
        else:
            final_preds = combined_preds

        # 4. Writing to file
        for term, score in final_preds:
            # Rounding and writing
            score_val = max(round(score, 3), 0.001)
            f.write(f"{pid}\t{term}\t{score_val:.3f}\n")

print(f"File {output_file} created using the 'Global Top-1500' strategy.")

--- Step 1: Generazione Candidati ---
Predizione CC...
Calcolo probabilità grezze (Threshold > 0.01)...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step

Predizione BP...
Calcolo probabilità grezze (Threshold > 0.01)...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step

Predizione MF...
Calcolo probabilità grezze (Threshold > 0.01)...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step

--- Step 2: Unione, Ordinamento e Taglio ---
✅ Fatto! File submission.tsv creato con la strategia 'Global Top-1500'.


# aggiustatina output (no questo ntebook)

In [4]:
from tqdm import tqdm # For the progress bar (optional but useful)

def load_go_relationships(obo_path):

    print(f"Reading ontology from {obo_path}...")
    term_parents = {}
    current_term = None

    with open(obo_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line == "[Term]":
                current_term = None
            elif line.startswith("id: GO:"):
                current_term = line.split("id: ")[1]
                if current_term not in term_parents:
                    term_parents[current_term] = []
            elif line.startswith("is_a: ") and current_term:
                # Extracts parent ID (e.g., is_a: GO:0008150 ! biological_process)
                parent_id = line.split("is_a: ")[1].split(" !")[0].strip()
                term_parents[current_term].append(parent_id)

    print(f"Ontology loaded: {len(term_parents)} terms found.")
    return term_parents

def check_hierarchy_consistency(submission_file, obo_path, max_errors_to_show=10):
    """
    Checks if the submission.tsv file respects the rule:
    Score(Parent) >= Score(Child)
    """
    # Load Parent-Child relationships
    go_parents = load_go_relationships(obo_path)

    # Load predictions
    df = pd.read_csv(submission_file, sep='\t', names=['id', 'term', 'score'])

    # Group by Protein to speed up checks
    violations_count = 0

    # Group the dataframe by protein ID
    grouped = df.groupby('id')

    for pid, group in tqdm(grouped, total=len(grouped)):
        # Create a {GO_ID: Score} map for this specific protein
        # This allows O(1) term score lookup
        scores_map = dict(zip(group['term'], group['score']))

        # Iterate over all predicted terms for this protein
        for child_term, child_score in scores_map.items():

            # Retrieve direct parents from the OBO file
            parents = go_parents.get(child_term, [])

            for parent in parents:
                # If the parent was predicted for this protein
                if parent in scores_map:
                    parent_score = scores_map[parent]

                    # CRITICAL CHECK: Parent must have score >= child
                    if parent_score < (child_score - 1e-6):
                        violations_count += 1
                        if violations_count <= max_errors_to_show:
                            print(f"VIOLATION Protein {pid}:")
                            print(f"   Child {child_term} (Score: {child_score:.4f}) > Parent {parent} (Score: {parent_score:.4f})")

                # NOTE: If the parent is NOT in the file, technically it has score 0.
                # So if the child has score > 0, it is technically a violation.
                # However, many submission files are 'sparse', so we mainly check
                # if the parent is present with a lower score.

    print("\n" + "="*40)
    print("CHECK RESULT")
    print("="*40)
    if violations_count == 0:
        print("EXCELLENT! No hierarchy violations detected.")
    else:
        print(f"Detected {violations_count} total hierarchy violations.")
        print("Suggestion: Use an 'enforce consistency' function (e.g., max propagation) to fix this.")

# --- EXECUTION ---
# Ensure paths are correct
obo_file_path = "go-basic.obo"   # Or the path where you saved the OBO
tsv_file_path = "submission.tsv" # The file you just generated

if os.path.exists(tsv_file_path) and os.path.exists(obo_file_path):
    check_hierarchy_consistency(tsv_file_path, obo_file_path)
else:
    print("Error: .obo or .tsv files not found.")

Reading ontology from go-basic.obo...
Ontology loaded: 47637 terms found.


 13%|█▎        | 134/1000 [00:00<00:01, 686.05it/s]

VIOLATION Protein A0A0B4JCV4:
   Child GO:0000235 (Score: 1.0000) > Parent GO:0005876 (Score: 0.9990)
VIOLATION Protein A0A0B4JCV4:
   Child GO:0000235 (Score: 1.0000) > Parent GO:0005881 (Score: 0.9990)
VIOLATION Protein A0A0B4JCV4:
   Child GO:0005818 (Score: 1.0000) > Parent GO:0099080 (Score: 0.9990)
VIOLATION Protein A0A0B4JCV4:
   Child GO:0010639 (Score: 1.0000) > Parent GO:0033043 (Score: 0.9990)
VIOLATION Protein A0A0B4JCV4:
   Child GO:0051129 (Score: 1.0000) > Parent GO:0048523 (Score: 0.9820)
VIOLATION Protein A0A0B4JCV4:
   Child GO:0051129 (Score: 1.0000) > Parent GO:0051128 (Score: 0.9970)
VIOLATION Protein A0A0B4JCV4:
   Child GO:0099081 (Score: 1.0000) > Parent GO:0099080 (Score: 0.9990)
VIOLATION Protein A0A0B4JCV4:
   Child GO:0051656 (Score: 0.9990) > Parent GO:0051640 (Score: 0.9980)
VIOLATION Protein A0A0B4JCV4:
   Child GO:1902903 (Score: 0.9990) > Parent GO:0051128 (Score: 0.9970)
VIOLATION Protein A0A0B4JCV4:
   Child GO:0033043 (Score: 0.9990) > Parent GO:0051

100%|██████████| 1000/1000 [00:00<00:00, 1002.53it/s]


CHECK RESULT
Detected 77801 total hierarchy violations.
Suggestion: Use an 'enforce consistency' function (e.g., max propagation) to fix this.


In [6]:
def enforce_consistency(input_file, output_file, obo_path):
    """
    Reads the file, corrects violated scores, and saves a new file.
    Logic: Score(Parent) = max(Score(Parent), Score(Child))
    """
    # 1. Load Ontology (Using the replaced function)
    go_parents = load_go_relationships(obo_path)

    # 2. Load Predictions
    print(f"Reading prediction file: {input_file}...")
    df = pd.read_csv(input_file, sep='\t', names=['id', 'term', 'score'])

    # 3. Processing
    print("Correcting hierarchy (Propagation in progress)...")

    corrected_rows = []

    # Group by protein (process one protein at a time)
    grouped = df.groupby('id')

    for pid, group in tqdm(grouped, total=len(grouped)):
        # Map {GO_ID: Score} for quick access
        scores = dict(zip(group['term'], group['score']))

        # ITERATIVE PROPAGATION
        # Continue cycling until scores stop changing (convergence)
        # This is needed because if I raise the parent, I might need to raise the grandparent, etc.
        changed = True
        while changed:
            changed = False
            # Iterate over a fixed list of current keys
            current_terms = list(scores.keys())

            for term in current_terms:
                current_score = scores[term]

                # Get parents
                parents = go_parents.get(term, [])

                for parent in parents:
                    parent_score = scores.get(parent, 0.0)

                    # IF CHILD SCORE > PARENT SCORE -> UPDATE PARENT
                    if current_score > parent_score:
                        scores[parent] = current_score
                        changed = True # We need another loop to propagate to grandparents

        # At the end of the while loop, the hierarchy for this protein is perfect.
        # Add rows to the final list
        for term, score in scores.items():
            corrected_rows.append({'id': pid, 'term': term, 'score': score})

    # Create Final DataFrame and Save
    print("Creating final file...")
    df_corrected = pd.DataFrame(corrected_rows)

    # Sort for cleanliness
    df_corrected = df_corrected.sort_values(by=['id', 'score'], ascending=[True, False])

    df_corrected.to_csv(output_file, sep='\t', index=False, header=False, float_format='%.3f')

    print(f"Corrected file saved as: {output_file}")
    print(f"   Original rows: {len(df)}")
    print(f"   Rows after correction (including added parents): {len(df_corrected)}")

# --- EXECUTION ---
input_tsv = "submission.tsv"
output_tsv = "submission_corrected.tsv"
obo_file = "go-basic.obo" # Or the correct path

if os.path.exists(input_tsv) and os.path.exists(obo_file):
    enforce_consistency(input_tsv, output_tsv, obo_file)
else:
    print("Files missing!")

Reading ontology from go-basic.obo...
Ontology loaded: 47637 terms found.
Reading prediction file: submission.tsv...
Correcting hierarchy (Propagation in progress)...


100%|██████████| 1000/1000 [00:00<00:00, 1040.10it/s]


Creating final file...
Corrected file saved as: submission_corrected.tsv
   Original rows: 357463
   Rows after correction (including added parents): 371516


In [9]:
def check_cafa_limit(file_path, limit=1500):
    """
    Checks if any protein in the submission file has more than 'limit' annotations.
    """
    print(f"Checking annotation counts in: {file_path}...")

    # Load the file
    # We assume standard CAFA format: ID <tab> Term <tab> Score
    df = pd.read_csv(file_path, sep='\t', names=['id', 'term', 'score'])

    # Count how many terms each protein has
    counts = df['id'].value_counts()

    # Find those exceeding the limit
    violators = counts[counts > limit]

    print("\n" + "="*40)
    print("LIMIT CHECK RESULTS")
    print("="*40)

    if len(violators) == 0:
        print(f"All {len(counts)} proteins have {limit} or fewer annotations.")
        print(f"   Max annotations found for a single protein: {counts.max()}")
    else:
        print(f"{len(violators)} proteins exceed the limit of {limit} annotations.")
        print("   Top violators (ID: Count):")
        print(violators.head(10))

        print("   Sort by score descending and keep top 1500 per protein.")

# --- EXECUTION ---
file_to_check = "submission_corrected.tsv" # The output of the consistency step

check_cafa_limit(file_to_check)

Checking annotation counts in: submission_corrected.tsv...

LIMIT CHECK RESULTS
All 1000 proteins have 1500 or fewer annotations.
   Max annotations found for a single protein: 1410
